In [5]:
def find_rst_headers(lines):
    headers = []
    for i in range(len(lines)):
        cur_line = lines[i].strip()

        if cur_line == "": continue

        # check if last line before peeking ahead
        if i+1 < len(lines):
            next_line = lines[i+1].strip()
            if next_line != "" and len(next_line) >= len(cur_line) and len(set(next_line)) == 1 and next_line[0] in ["=", "-", "~", "^", '"', "'", "\\", ".", "*", "+", "#", ":"]:
                headers.append([cur_line, i])

    return headers


In [6]:
test_lines = [
    "Installing Requests",
    "===================",
    "",
    "Some body text here.",
    "",
    "A Subsection",
    "-------------",
    "More text."
]
print(find_rst_headers(test_lines))

[['Installing Requests', 0], ['A Subsection', 5]]


In [7]:
print(len("Installing Requests"))
print(len("=================="))

19
18


In [12]:
def extract_sections(lines, headers, header_line_count):
    sections = []
    for h in range(len(headers)):
        header_text = headers[h][0]
        header_index = headers[h][1]
        body_start = header_index + header_line_count  #skip past header text + underline if rst

        if h+1 >= len(headers):
            body_end = len(lines) #no next header, so go to the end of the file
        else:
            body_end = headers[h+1][1]  # stop right before the next header starts
        body_lines = lines[body_start : body_end]
        body_text = "\n".join(body_lines).strip()

        sections.append([header_text, body_text])

    return sections

In [13]:
test_lines = [
    "Installing Requests",
    "====================",
    "This is line one of the body.",
    "This is line two of the body.",
    "",
    "A Subsection",
    "-------------",
    "Body text for the subsection.",
    "More subsection text."
]

test_headers = find_rst_headers(test_lines)
print(test_headers)

sections = extract_sections(test_lines, test_headers, 2)
for s in sections:
    print(s)

[['Installing Requests', 0], ['A Subsection', 5]]
['Installing Requests', 'This is line one of the body.\nThis is line two of the body.']
['A Subsection', 'Body text for the subsection.\nMore subsection text.']


In [14]:
def find_markdown_headers(lines):
    headers = []
    for i in range(len(lines)):
        cur_line = lines[i]
        if cur_line == "":
            continue
        if cur_line.startswith("#"):
            level = len(cur_line) - len(cur_line.lstrip("#"))
            header_text = cur_line.strip().strip("#").strip()
            headers.append([header_text, i, level])
    return headers

In [15]:
test_lines = [
    "# Installing httpx",
    "",
    "Some intro text.",
    "## Quickstart",
    "More text here."
]
print(find_markdown_headers(test_lines))

[['Installing httpx', 0, 1], ['Quickstart', 3, 2]]


In [17]:
test_lines = [
    "# Installing httpx",
    "This is line one of the intro.",
    "This is line two of the intro.",
    "",
    "## Quickstart",
    "Body text for quickstart.",
    "More quickstart text."
]

test_headers = find_markdown_headers(test_lines)
print(test_headers)

sections = extract_sections(test_lines, test_headers, 1)
for s in sections:
    print(s)

[['Installing httpx', 0, 1], ['Quickstart', 4, 2]]
['Installing httpx', 'This is line one of the intro.\nThis is line two of the intro.']
['Quickstart', 'Body text for quickstart.\nMore quickstart text.']


In [18]:
from pathlib import Path
print(Path("quickstart.v2.md").suffix)
print(Path("installation.rst").suffix)

.md
.rst


In [19]:
def find_files(root_dir):
    file_paths = []
    for path in Path(root_dir).rglob("*"):
        if path.suffix in [".md", ".rst"]:
            file_paths.append(path)

    return file_paths

def process_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

        if filepath.suffix == ".rst":
            headers = find_rst_headers(lines)
            header_line_count = 2
        elif filepath.suffix == ".md":
            headers = find_markdown_headers(lines)
            header_line_count = 1
        else:
            raise ValueError(f"Unsupported file type: {filepath.suffix}")

        sections = extract_sections(lines, headers, header_line_count)
        return sections

In [22]:
from pathlib import Path

# Since the notebook lives in notebooks/, go up one level to reach data/raw
files = find_files("../data/raw")
print(f"Found {len(files)} files")
print(files[:5])  # peek at a few paths

if files:
    filepath = files[0]
    print(f"\nTesting on: {filepath}")
    print(f"Suffix: {filepath.suffix}")

    sample_sections = process_file(filepath)
    print(f"\nFound {len(sample_sections)} sections in this file\n")

    for s in sample_sections[:2]:
        print(s)
        print("---")
else:
    print("Still found 0 files — the path is wrong, or data/raw is empty/misplaced.")

Found 38 files
[WindowsPath('../data/raw/httpx/docs/api.md'), WindowsPath('../data/raw/httpx/docs/async.md'), WindowsPath('../data/raw/httpx/docs/code_of_conduct.md'), WindowsPath('../data/raw/httpx/docs/compatibility.md'), WindowsPath('../data/raw/httpx/docs/contributing.md')]

Testing on: ..\data\raw\httpx\docs\api.md
Suffix: .md

Found 10 sections in this file

['Developer Interface', '']
---
['Helper Functions', "!!! note\n\n    Only use these functions if you're testing HTTPX in a console\n\n    or making a small number of requests. Using a `Client` will\n\n    enable HTTP/2 and connection pooling for more efficient and\n\n    long-lived connections.\n\n\n\n::: httpx.request\n\n    :docstring:\n\n\n\n::: httpx.get\n\n    :docstring:\n\n\n\n::: httpx.options\n\n    :docstring:\n\n\n\n::: httpx.head\n\n    :docstring:\n\n\n\n::: httpx.post\n\n    :docstring:\n\n\n\n::: httpx.put\n\n    :docstring:\n\n\n\n::: httpx.patch\n\n    :docstring:\n\n\n\n::: httpx.delete\n\n    :docstring:\n